# 00 · Conceptos de Selenium

**Máster Universitario en Ciberseguridad · Ciberinteligencia · Taller OSINT**

Pruebas mínimas de cada concepto de las diapositivas. Usamos **quotes.toscrape.com**, un sitio creado precisamente para practicar scraping: así los ejemplos no dependen de que una web real cambie su diseño.

| Sección | Concepto | Diapositiva |
|---|---|---|
| 1 | requests frente a un navegador | requests frente a un navegador |
| 2 | El ciclo de un script | Cómo funciona Selenium |
| 3 | Localizadores | Localizadores |
| 4 | Esperas | Esperas |
| 5 | Interactuar: clic, scroll, visible ≠ existe | Interactuar con la página |
| 6 | Errores típicos | Errores típicos |
| 7 | robots.txt | Límites |

Ejecuta las celdas en orden (`Mayús + Intro`).

## 0 · Preparar el entorno

In [2]:
# Instala Selenium y Google Chrome en la máquina de Colab (tarda 1-2 minutos)
!pip -q install selenium
!apt-get -qq update > /dev/null
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null
!rm -f google-chrome-stable_current_amd64.deb
!google-chrome --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 9.7 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Google Chrome 154.0.8037.57 


In [10]:
import selenium
from selenium import webdriver

def crear_driver():
    """Abre un Chrome sin ventana (headless), preparado para funcionar en Colab."""
    opciones = webdriver.ChromeOptions()
    for arg in ["--headless=new",          # sin ventana
                "--no-sandbox",            # necesario en Colab
                "--disable-dev-shm-usage", # evita problemas de memoria compartida en Colab
                "--disable-gpu",
                "--window-size=1920,1080", # tamaño de pantalla simulado (afecta a qué es 'visible')
                "--lang=es-ES"]:
        opciones.add_argument(arg)
    # Desde Selenium 4.6, Selenium Manager descarga automáticamente el chromedriver
    # que corresponde a la versión de Chrome instalada: ya no hace falta webdriver-manager.
    return webdriver.Chrome(options=opciones)

print("Selenium", selenium.__version__)

Selenium 4.49.0


In [11]:
import time, re, os, json, hashlib
from datetime import datetime, timezone, date, timedelta

import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (TimeoutException, WebDriverException,
                                        NoSuchElementException, ElementClickInterceptedException)

## 1 · requests frente a un navegador

La misma web tiene dos versiones. En la normal, las citas vienen escritas en el HTML. En `/js`, el HTML solo trae un script, y es el navegador quien pinta las citas al ejecutarlo.

`requests` descarga el HTML pero **no ejecuta JavaScript**. Selenium usa un navegador real, que sí lo ejecuta.

In [12]:
import requests
from bs4 import BeautifulSoup

for url in ["https://quotes.toscrape.com/", "https://quotes.toscrape.com/js/"]:
    html = requests.get(url, timeout=15).text
    n = len(BeautifulSoup(html, "html.parser").select("div.quote"))
    print(f"requests  → {url:35} {n} citas")

requests  → https://quotes.toscrape.com/        10 citas
requests  → https://quotes.toscrape.com/js/     0 citas


In [13]:
wd = crear_driver()
wd.get("https://quotes.toscrape.com/js/")
citas = WebDriverWait(wd, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.quote")))
print(f"Selenium  → https://quotes.toscrape.com/js/     {len(citas)} citas")

Selenium  → https://quotes.toscrape.com/js/     10 citas


**Conclusión:** antes de usar Selenium, comprueba si `requests` ya ve lo que necesitas. Es más rápido, más ligero y más fácil de mantener.

## 2 · El ciclo de un script: abrir, localizar, actuar, extraer, cerrar

In [14]:
wd.get("https://quotes.toscrape.com/")                                  # 1. Abrir
primera = wd.find_element(By.CSS_SELECTOR, "div.quote")                 # 2. Localizar
wd.execute_script("arguments[0].scrollIntoView();", primera)            # 3. Actuar
texto = primera.find_element(By.CSS_SELECTOR, "span.text").text         # 4. Extraer
autor = primera.find_element(By.CSS_SELECTOR, "small.author").text
print(f"{texto}\n  — {autor}")
# 5. Cerrar: wd.quit() (lo hacemos al final del notebook)

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
  — Albert Einstein


## 3 · Localizadores

Tres formas de llegar al mismo sitio. Después, la diferencia entre un XPath **absoluto** (depende de la posición exacta de cada etiqueta) y uno **relativo** (busca el elemento por lo que es).

In [15]:
autores_css   = wd.find_elements(By.CSS_SELECTOR, "small.author")
autores_xpath = wd.find_elements(By.XPATH, "//small[@class='author']")
etiquetas     = wd.find_elements(By.CLASS_NAME, "tag")
siguiente     = wd.find_element(By.PARTIAL_LINK_TEXT, "Next")

print("CSS  :", [a.text for a in autores_css[:3]])
print("XPath:", [a.text for a in autores_xpath[:3]])
print("Etiquetas en la página:", len(etiquetas))
print("Enlace 'Next' apunta a:", siguiente.get_attribute("href"))

CSS  : ['Albert Einstein', 'J.K. Rowling', 'Albert Einstein']
XPath: ['Albert Einstein', 'J.K. Rowling', 'Albert Einstein']
Etiquetas en la página: 40
Enlace 'Next' apunta a: https://quotes.toscrape.com/page/2/


## 4 · Esperas: a una condición, no a un reloj

`/js-delayed/` tarda varios segundos en pintar las citas. Primero lo intentamos con una espera fija demasiado corta; después, con una espera explícita.

In [13]:
wd.get("https://quotes.toscrape.com/js-delayed/")
time.sleep(2)                                        # espera fija: 2 segundos, pase lo que pase
print("Con time.sleep(2):", len(wd.find_elements(By.CSS_SELECTOR, "div.quote")), "citas")

Con time.sleep(2): 10 citas


In [14]:
wd.get("https://quotes.toscrape.com/js-delayed/")
inicio = time.time()
citas = WebDriverWait(wd, 30).until(                 # espera explícita: hasta 30 s, sigue en cuanto aparecen
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.quote")))
print(f"Con WebDriverWait: {len(citas)} citas tras {time.time() - inicio:.1f} s")

Con WebDriverWait: 10 citas tras 10.2 s


## 5 · Interactuar con la página

### 5.1 Clic, con scroll previo

In [16]:
wd.get("https://quotes.toscrape.com/")
boton = wd.find_element(By.CSS_SELECTOR, "li.next > a")
wd.execute_script("arguments[0].scrollIntoView({block: 'center'});", boton)   # llevarlo a la vista
try:
    boton.click()
except ElementClickInterceptedException:
    wd.execute_script("arguments[0].click();", boton)                        # plan B: clic con JavaScript
print("Ahora estamos en:", wd.current_url)

Ahora estamos en: https://quotes.toscrape.com/page/2/


### 5.2 Scroll infinito

`/scroll` carga más citas a medida que bajas, como hacen las redes sociales.

In [16]:
wd.get("https://quotes.toscrape.com/scroll")
WebDriverWait(wd, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.quote")))
for vuelta in range(3):
    antes = len(wd.find_elements(By.CSS_SELECTOR, "div.quote"))
    wd.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    try:
        WebDriverWait(wd, 10).until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "div.quote")) > antes)
    except TimeoutException:
        print("No carga más")
        break
    print(f"Scroll {vuelta + 1}: {antes} → {len(wd.find_elements(By.CSS_SELECTOR, 'div.quote'))} citas")

Scroll 1: 10 → 20 citas
Scroll 2: 20 → 30 citas
Scroll 3: 30 → 40 citas


## 6 · Errores típicos

Provocamos tres a propósito. Fíjate en que **capturamos la excepción y mostramos el mensaje**: nunca `except: pass`.

In [ ]:
# NoSuchElementException: el localizador no encuentra nada
try:
    wd.find_element(By.ID, "esto-no-existe")
except NoSuchElementException as e:
    print("NoSuchElementException →", str(e).splitlines()[0][:90])

In [ ]:
# TimeoutException: la espera llega al máximo sin que se cumpla la condición
try:
    WebDriverWait(wd, 3).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".no-existe")))
except TimeoutException:
    print("TimeoutException → la condición no se cumplió en 3 segundos")

In [ ]:
# StaleElementReferenceException: la página cambió después de localizar el elemento
from selenium.common.exceptions import StaleElementReferenceException
cita = wd.find_element(By.CSS_SELECTOR, "div.quote span.text")
wd.refresh()
try:
    print(cita.text)
except StaleElementReferenceException:
    print("StaleElementReferenceException → hay que volver a localizar el elemento")

`ElementClickInterceptedException` (algo tapa el elemento) es difícil de provocar aquí. Aparece con banners de cookies o anuncios: la solución es el scroll previo o el clic con JavaScript de la sección 5.1.

## 7 · Límites: robots.txt

`robots.txt` indica qué rutas pide el sitio que no recorran los programas automáticos. No es una ley, pero ignorarlo es una mala práctica. Los **términos de uso** de cada web son otra cosa y hay que leerlos aparte.

In [17]:
from urllib.robotparser import RobotFileParser

def consultar_robots(sitio, ruta):
    rp = RobotFileParser(sitio.rstrip("/") + "/robots.txt")
    try:
        rp.read()
        print(f"{sitio}{ruta} → ¿permitido?", rp.can_fetch("*", sitio.rstrip("/") + ruta))
    except Exception as e:
        print(f"{sitio}: no se pudo leer robots.txt ({type(e).__name__})")

consultar_robots("https://quotes.toscrape.com", "/js/")
consultar_robots("https://getdaytrends.com", "/es/spain/")

https://quotes.toscrape.com/js/ → ¿permitido? True
https://getdaytrends.com/es/spain/ → ¿permitido? False


In [ ]:
# Mirarlo a mano también es sencillo:
print(requests.get("https://getdaytrends.com/robots.txt", timeout=15).text[:800])

## Cerrar el navegador

In [ ]:
wd.quit()
print("Navegador cerrado")